# 03 — LLM Baseline

**Purpose**: Establish what a language model can predict about CDR worsening when given visit 1 and visit 2 data serialised as text.

All logic lives in `src/llm_predictor.py`. This notebook calls those functions and inspects the outputs.

**Key limitation being demonstrated**: The LLM has no internal model of how Alzheimer's progression works physiologically. It matches the text description of this patient against patterns in its training data. It cannot reason about the *rate and pattern* of change across visits — it sees a snapshot description.

In [ ]:
import sys
from pathlib import Path

project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import os
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

from src.llm_predictor import (
    build_clinical_prompt,
    predict_all_test_participants,
    summarise_llm_predictions,
    save_llm_predictions,
)

load_dotenv(project_root / ".env")

RESULTS_DIR = project_root / "results"

print(f"Project root : {project_root}")

## 1. Load test set

In [ ]:
test_df = pd.read_csv(project_root / "data" / "processed" / "test_participants.csv")
print(f"Test participants: {len(test_df)}")
test_df[["subject_id", "participant_group", "mmse_v1", "mmse_v2", "cdr_v1", "cdr_v2", "cdr_worsened_after_v2"]].head(10)

## 2. Inspect the prompt

Before running 29 API calls, inspect what the LLM actually sees for one participant.

In [ ]:
sample_participant = test_df.iloc[0]
sample_prompt = build_clinical_prompt(sample_participant)
print(sample_prompt)

## 3. Run predictions

In [ ]:
openai_client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
test_with_predictions = predict_all_test_participants(test_df, openai_client)

## 4. Summary

In [ ]:
summarise_llm_predictions(test_with_predictions)

In [ ]:
# Inspect prediction vs actual for every participant
test_with_predictions[[
    "subject_id", "participant_group",
    "mmse_v1", "mmse_v2", "mmse_delta_v1_v2",
    "cdr_v1", "cdr_v2",
    "cdr_worsened_after_v2", "llm_prediction"
]]

## 5. Save results

In [ ]:
save_llm_predictions(test_with_predictions, RESULTS_DIR)